# 02 Analysis Dashboard

역할: `01_run_orchestrator.ipynb`가 생성한 summary/log artifact를 읽어서 비교 테이블, ranking, 분포 plot을 확인한다.
하지 않는 일: dataset bootstrap, runtime setup, runner 실행, 보고서용 파일 저장.


## Cell Role: Analysis Path and Helper Setup

저장된 repo를 기준으로 path를 잡고 dashboard loader/helper를 import한다.
이 노트북은 runner를 실행하지 않으므로 runtime setup helper는 불러오지 않는다.


In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name == 'ReGraM'), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']
SRC_ROOT = EXP_ROOT / 'src'
CORE_SRC = SRC_ROOT / 'core'
for source_root in (SRC_ROOT, CORE_SRC):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

for module_name in ('orchestration.notebook_orchestration', 'orchestration.dashboard_loader'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from orchestration.dashboard_loader import (
    display_dashboard_tables,
    display_wandb_panel_recommendations,
    load_dashboard_frames,
    render_visual_dashboard,
)
from orchestration.notebook_orchestration import (
    SEVERITY_ORDER,
    build_baseline_specs,
    build_requested_run_configs,
    discover_manifest_names,
    display_environment_summary,
    display_title,
    resolve_manifest_paths,
)

RAW_LOCO_ROOT = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection'
UNIVAD_CAPTION_DATA_ROOT = REPO_ROOT / 'data' / 'mvtec_loco_caption'
BASELINE_SPECS = build_baseline_specs(
    repo_root=REPO_ROOT,
    exp_root=EXP_ROOT,
    raw_loco_root=RAW_LOCO_ROOT,
    univad_caption_data_root=UNIVAD_CAPTION_DATA_ROOT,
    default_patchcore_device='cuda',
)
display_environment_summary(REPO_ROOT, EXP_ROOT, REPORT_ROOT)


## Cell Role: Load Summary Frames

분석 대상 baseline/category/manifest를 지정하고, 해당 summary JSON들을 DataFrame 묶음으로 로드한다.
여기 설정은 보통 01의 실행 범위와 맞춘다.


In [ ]:
DASHBOARD_BASELINES = ['UniVAD', 'PatchCore']
CATEGORIES = ['breakfast_box', 'screw_bag']
CONFIGURED_MANIFEST_NAMES = ['query_motion_blur.jsonl', 'query_low_light.jsonl', 'query_gaussian_noise.jsonl']
SELECTED_SEVERITIES = []  # Must match 01; [] means all. Examples: ['low'], ['medium'], ['high']
AUTO_DISCOVER_MANIFESTS = False
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
WANDB_PROJECT = 'regram-condition-shift'
WANDB_MODE = 'online'
TOP_K_WORST_SHIFTS = 10

ACTIVE_MANIFEST_NAMES = discover_manifest_names(
    manifest_roots=MANIFEST_ROOT_CANDIDATES,
    configured_names=CONFIGURED_MANIFEST_NAMES,
    auto_discover=AUTO_DISCOVER_MANIFESTS,
    excluded_names=EXCLUDED_MANIFEST_NAMES,
)
manifest_paths = resolve_manifest_paths(ACTIVE_MANIFEST_NAMES, MANIFEST_ROOT_CANDIDATES)
summary_sources = build_requested_run_configs(
    active_baselines=DASHBOARD_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
    selected_severities=SELECTED_SEVERITIES,
)
dashboard_frames = load_dashboard_frames(
    summary_sources=summary_sources,
    dashboard_baselines=DASHBOARD_BASELINES,
    categories=CATEGORIES,
    report_root=REPORT_ROOT,
    severity_order=SEVERITY_ORDER,
    top_k_worst_shifts=TOP_K_WORST_SHIFTS,
)
display_title('Loaded Summary Sources')
display(pd.DataFrame([
    {
        'baseline': config['baseline'],
        'category': config['category'],
        'summary_exists': config['summary_path'].exists(),
        'severities': ', '.join(config.get('selected_severities', [])) or 'all',
        'summary_path': str(config['summary_path']),
    }
    for config in summary_sources
]))


## Cell Role: Analysis Tables

artifact coverage, clean summary, shift summary, worst-case ranking을 표로 확인한다.
표는 해석용이며 새로운 실험 산출물을 만들지는 않는다.


In [ ]:
display_dashboard_tables(
    dashboard_frames,
    run_history=[],
    dashboard_baselines=DASHBOARD_BASELINES,
    top_k=TOP_K_WORST_SHIFTS,
)
display_wandb_panel_recommendations()


## Cell Role: Visual Dashboard

점수 분포, heatmap, gallery를 notebook 안에서 탐색적으로 렌더링한다.
저장용 figure가 필요하면 03 노트북을 사용한다.


In [ ]:
SHOW_VISUAL_DASHBOARD = True
if SHOW_VISUAL_DASHBOARD:
    render_visual_dashboard(dashboard_frames, dashboard_baselines=DASHBOARD_BASELINES)
